In [1]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from pathlib import Path

In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to /home/huzaifa/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/huzaifa/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/huzaifa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/huzaifa/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/huzaifa/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/huzaifa/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/huzaifa/nltk_data...
[nltk_data]   Package averaged_perceptron_ta

True

In [3]:
raw_data_path = Path("../data/raw/reviews.csv")
df = pd.read_csv(raw_data_path)
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [4]:
df.shape

(23486, 11)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Unnamed: 0               23486 non-null  int64
 1   Clothing ID              23486 non-null  int64
 2   Age                      23486 non-null  int64
 3   Title                    19676 non-null  str  
 4   Review Text              22641 non-null  str  
 5   Rating                   23486 non-null  int64
 6   Recommended IND          23486 non-null  int64
 7   Positive Feedback Count  23486 non-null  int64
 8   Division Name            23472 non-null  str  
 9   Department Name          23472 non-null  str  
 10  Class Name               23472 non-null  str  
dtypes: int64(6), str(5)
memory usage: 2.0 MB


In [6]:
df = df[['Review Text', 'Rating']].rename(columns={'Review Text': 'review', 'Rating': 'rating'}).dropna(subset=['review']).reset_index(drop=True)
df.head()

,review,rating
0,Absolutely wonderful - silky and sexy and comf...,4
1,Love this dress! it's sooo pretty. i happene...,5
2,I had such high hopes for this dress and reall...,3
3,"I love, love, love this jumpsuit. it's fun, fl...",5
4,This shirt is very flattering to all due to th...,5


In [7]:
original_text = df.iloc[1, 0]
original_text

'Love this dress!  it\'s sooo pretty.  i happened to find it in a store, and i\'m glad i did bc i never would have ordered it online bc it\'s petite.  i bought a petite and am 5\'8".  i love the length on me- hits just a little below the knee.  would definitely be a true midi on someone who is truly petite.'

In [8]:
def clean_text(text):
    if not text:
        return ""

    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

cleaned_text = clean_text(original_text)
cleaned_text

'Love this dress its sooo pretty i happened to find it in a store and im glad i did bc i never would have ordered it online bc its petite i bought a petite and am i love the length on me hits just a little below the knee would definitely be a true midi on someone who is truly petite'

In [9]:
def normalize_text(text):
    if not text:
        return ""

    return text.lower()


normalized_text = normalize_text(cleaned_text)
normalized_text

'love this dress its sooo pretty i happened to find it in a store and im glad i did bc i never would have ordered it online bc its petite i bought a petite and am i love the length on me hits just a little below the knee would definitely be a true midi on someone who is truly petite'

In [10]:
def tokenize_text(text):
    if not text:
        return []

    tokens = word_tokenize(text)
    return tokens

tokens = tokenize_text(normalized_text)
print(tokens)

['love', 'this', 'dress', 'its', 'sooo', 'pretty', 'i', 'happened', 'to', 'find', 'it', 'in', 'a', 'store', 'and', 'im', 'glad', 'i', 'did', 'bc', 'i', 'never', 'would', 'have', 'ordered', 'it', 'online', 'bc', 'its', 'petite', 'i', 'bought', 'a', 'petite', 'and', 'am', 'i', 'love', 'the', 'length', 'on', 'me', 'hits', 'just', 'a', 'little', 'below', 'the', 'knee', 'would', 'definitely', 'be', 'a', 'true', 'midi', 'on', 'someone', 'who', 'is', 'truly', 'petite']


In [11]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    if tag.startswith('V'):
        return wordnet.VERB
    if tag.startswith('N'):
        return wordnet.NOUN
    if tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

def lemmatize_tokens(tokens):
    if not tokens:
        return []

    lemmatizer = WordNetLemmatizer()
    pos_tags = nltk.pos_tag(tokens)
    lemmatized_tokens = [lemmatizer.lemmatize(token, get_wordnet_pos(pos)) for token, pos in pos_tags]
    return lemmatized_tokens

lemmatized_tokens = lemmatize_tokens(tokens)
print(lemmatized_tokens)

['love', 'this', 'dress', 'it', 'sooo', 'pretty', 'i', 'happen', 'to', 'find', 'it', 'in', 'a', 'store', 'and', 'im', 'glad', 'i', 'do', 'bc', 'i', 'never', 'would', 'have', 'order', 'it', 'online', 'bc', 'it', 'petite', 'i', 'buy', 'a', 'petite', 'and', 'be', 'i', 'love', 'the', 'length', 'on', 'me', 'hit', 'just', 'a', 'little', 'below', 'the', 'knee', 'would', 'definitely', 'be', 'a', 'true', 'midi', 'on', 'someone', 'who', 'be', 'truly', 'petite']


In [12]:
def stopword_removal(tokens):
    if not tokens:
        return []

    stop_words = set(stopwords.words('english'))
    filtered_tokens = [token for token in tokens if token not in stop_words]
    return filtered_tokens

filtered_tokens = stopword_removal(tokens)
print(filtered_tokens)

['love', 'dress', 'sooo', 'pretty', 'happened', 'find', 'store', 'im', 'glad', 'bc', 'never', 'would', 'ordered', 'online', 'bc', 'petite', 'bought', 'petite', 'love', 'length', 'hits', 'little', 'knee', 'would', 'definitely', 'true', 'midi', 'someone', 'truly', 'petite']


In [13]:
def preprocess_text(text):
    if not text:
        return ""

    cleaned_text = clean_text(text)
    normalized_text = normalize_text(cleaned_text)
    tokens = tokenize_text(normalized_text)
    lemmatized_tokens = lemmatize_tokens(tokens)
    filtered_tokens = stopword_removal(lemmatized_tokens)

    return ' '.join(filtered_tokens)

preprocess_text(original_text)

'love dress sooo pretty happen find store im glad bc never would order online bc petite buy petite love length hit little knee would definitely true midi someone truly petite'

In [14]:
df['review'] = df['review'].apply(lambda x: preprocess_text(x))
df.head()

,review,rating
0,absolutely wonderful silky sexy comfortable,4
1,love dress sooo pretty happen find store im gl...,5
2,high hope dress really want work initially ord...,3
3,love love love jumpsuit fun flirty fabulous ev...,5
4,shirt flattering due adjustable front tie perf...,5


In [16]:
processed_data_dir = Path("../data/processed/")
processed_data_dir.mkdir(parents=True, exist_ok=True)
processed_data_path = processed_data_dir / "processed.csv"
df.to_csv(processed_data_path, index=False)